## <span style=color:blue>Data Acquisition & Cleaning:</span>

### <span style=color:green>Importing Packages</span>

In [1]:
#Packages/file:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chisquare
from scipy.stats import chi2_contingency
from scipy.stats import spearmanr
from scipy.stats import kruskal
df = pd.read_csv("ARS_data.csv")

### <span style=color:green>Inspecting Data</span>

In [2]:
df.head(3)

,Date,Have you taken this survey before?,Donation Contents,Other Items,Accessories,Bags,Books,Cups,Dresses,Electronics,...,Race,Affliation,School Year,College,Faculty Role,Faculty Role (Other),Outhreach Method,Outreaach (Other),Visit Frequency,Will Repeat Donation
0,2026-01-16 17:56:48,No,"Electronics,School supplies",NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,Prefer not to answer,Community member,NaN,NaN,NaN,NaN,Online website,NaN,I have never visited,Maybe
1,2026-01-20 12:38:19,No,"Long-sleeves,Pants,T-shirts,Tank tops",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Prefer not to answer,UC Davis student,Second Year,Agriculture and Environmental Sciences,NaN,NaN,Volunteer/staff,NaN,More than once a week,Yes
2,2026-01-22 16:16:59,No,Household goods,Clothes Hangers,NaN,NaN,NaN,NaN,NaN,NaN,...,Prefer not to answer,UC Davis student,Graduate Student,Engineering,NaN,NaN,Online website,NaN,I have never visited,Maybe


In [3]:
df.columns

Index(['Date', 'Have you taken this survey before?', 'Donation Contents',
       'Other Items', 'Accessories', 'Bags', 'Books', 'Cups', 'Dresses',
       'Electronics', 'Hats', 'Household Goods', 'Jackets', 'Jewelry',
       'Long-Sleeves', 'Pants', 'School Supplies', 'Shoes', 'Shorts', 'Skirts',
       'Sweaters', 'Shirts', 'Tank Tops', 'Water Bottles', 'Other Items (#)',
       'Total Items', 'Condition (New/Like-New)',
       'Condition (Moderately-Used)', 'Donation Reason',
       'Donation Reason (Other)', 'Gender', 'Ethnicity', 'Race', 'Affliation',
       'School Year', 'College', 'Faculty Role', 'Faculty Role (Other)',
       'Outhreach Method', 'Outreaach (Other)', 'Visit Frequency',
       'Will Repeat Donation'],
      dtype='object')

### <span style=color:green>String Splitting "Donation Reason"</span>

In [4]:
df["Donation Reason"].dtype

dtype('O')

In [5]:
df_q1 = df.copy()

df_q1["Donation Reason"] = df_q1["Donation Reason"].astype(str)

# Splits all reasons by a ","
df_q1["Donation Reason"] = df_q1["Donation Reason"].str.split(",")

# Explodes the list into individual rows specifically for the Q1 dataframe
df_q1 = df_q1.explode("Donation Reason")

# Removes white space
df_q1["Donation Reason"] = df_q1["Donation Reason"].str.strip()

### <span style=color:green>Ennumerating "Frequency of Visits"</span>

In [6]:
#Maps each value scaled on visit frequency.
visit_mapping = {
    "More than once a week": 6,
    "About once a week": 5,
    "About once per month": 4,
    "About once per quarter": 3,
    "Less than once a quarter": 2,
    "I have never visited": 1
}
#Update Visit col.
df["Visits"] = df["Visit Frequency"].map(visit_mapping)
df = df.dropna(subset=["Visits"]).copy()
df["Visits"]

0     1.0
1     6.0
2     1.0
3     3.0
4     6.0
5     3.0
6     6.0
7     3.0
8     1.0
9     1.0
10    2.0
11    2.0
12    1.0
13    1.0
14    3.0
15    1.0
16    1.0
17    1.0
18    4.0
19    5.0
20    2.0
21    2.0
22    2.0
24    4.0
26    2.0
27    3.0
28    2.0
30    1.0
31    5.0
32    3.0
33    2.0
34    3.0
36    1.0
37    4.0
38    4.0
39    6.0
40    5.0
41    3.0
43    1.0
44    2.0
45    6.0
46    1.0
48    3.0
49    4.0
50    4.0
Name: Visits, dtype: float64

In [7]:
df["Total Items"] = df["Total Items"].astype(int)

# <span style=color:blue>Chi-Squared Test:</span>

### <span style=color:green>**Question 1:** Is there an association between donor type(student, staff, community) and their motivations to donate?</span>

In [8]:
#Creates Contingency Table:
cont_table = pd.crosstab(df_q1['Affliation'], df_q1['Donation Reason'])

print(cont_table)

Donation Reason   Charity  Declutter home  Emotional fulfillment  \
Affliation                                                         
Community member        2               6                      1   
UC Davis staff          1               2                      0   
UC Davis student        9              25                      8   

Donation Reason   No longer affiliated with the company/job/college of clothing  \
Affliation                                                                        
Community member                                                  0               
UC Davis staff                                                    0               
UC Davis student                                                  4               

Donation Reason   Other:  Support environmental sustainability  \
Affliation                                                       
Community member       0                                     6   
UC Davis staff         0                     

In [9]:
#pulling specific statistics:
chi2, p, dof, expected = chi2_contingency(cont_table)

print("Chi-square:", chi2) #test stat
print("p-value:", p)

Chi-square: 3.562058080808081
p-value: 0.9901074723534397


# <span style=color:blue>Spearman's Rank Correlation:</span>

### <span style=color:green>**Question 2:** Is there a correlation between visits to Aggie Reuse and the number of items donated? </span>

In [10]:
df.columns

Index(['Date', 'Have you taken this survey before?', 'Donation Contents',
       'Other Items', 'Accessories', 'Bags', 'Books', 'Cups', 'Dresses',
       'Electronics', 'Hats', 'Household Goods', 'Jackets', 'Jewelry',
       'Long-Sleeves', 'Pants', 'School Supplies', 'Shoes', 'Shorts', 'Skirts',
       'Sweaters', 'Shirts', 'Tank Tops', 'Water Bottles', 'Other Items (#)',
       'Total Items', 'Condition (New/Like-New)',
       'Condition (Moderately-Used)', 'Donation Reason',
       'Donation Reason (Other)', 'Gender', 'Ethnicity', 'Race', 'Affliation',
       'School Year', 'College', 'Faculty Role', 'Faculty Role (Other)',
       'Outhreach Method', 'Outreaach (Other)', 'Visit Frequency',
       'Will Repeat Donation', 'Visits'],
      dtype='object')

In [11]:
rho, p = spearmanr(df["Visits"],
                   df["Total Items"])

print("Spearman rho:", rho) #test stat 
print("p-value:", p)

Spearman rho: -0.4443937299026817
p-value: 0.0022263235991933383


# <span style=color:blue>**Part 3:** Kruskal–Wallis H Test</span>

### <span style=color:green>**Question 3:** Is there a difference in student years and the number of items they donate? </span>

In [12]:
df.groupby("School Year")["Total Items"].describe()

,count,mean,std,min,25%,50%,75%,max
School Year,,,,,,,,
Fifth Year or above,1.0,8.000000,NaN,8.0,8.0,8.0,8.00,8.0
First Year,12.0,5.083333,3.502164,1.0,3.0,4.0,7.00,13.0
Fourth Year,10.0,24.800000,24.561940,2.0,5.0,19.5,37.75,80.0
Graduate Student,3.0,10.000000,13.000000,2.0,2.5,3.0,14.00,25.0
Second Year,6.0,5.500000,4.929503,1.0,2.0,3.5,8.75,13.0
Third Year,3.0,6.000000,3.000000,3.0,4.5,6.0,7.50,9.0


In [13]:
#grouping total items by school year:
groups = [
    group["Total Items"]
    for name, group in df.groupby("School Year")
]

stat, p = kruskal(*groups)

print("H statistic:", stat) #state stat
print("p-value:", p)

H statistic: 6.756893629043683
p-value: 0.23935730009037248


### <span style=color:green>**Question 4:** Is there an association between item quality(new/light use, moderate use) and item category?</span> (Work in Progress)

In [14]:
# # Simple mapping function
# def condition_to_numeric(x):
#     try:
#         return float(x)  # if it's already a number
#     except:
#         # Map text descriptions to approximate counts
#         if isinstance(x, str):
#             if "Null" in x.lower():
#                 return 0
#             elif "A few items" in x.lower():
#                 return 2
#             elif "A good amount" in x.lower():
#                 return 5
#         return 0  # if empty or unrecognized

# df['Condition (New/Like-New)'] = df['Condition (New/Like-New)'].apply(condition_to_numeric)
# df['Condition (Moderately-Used)'] = df['Condition (Moderately-Used)'].apply(condition_to_numeric)

In [15]:
# categories = ['Accessories','Bags','Books','Cups','Dresses','Electronics','Hats',
#               'Household Goods','Jackets','Jewelry','Long-Sleeves','Pants','School Supplies',
#               'Shoes','Shorts','Skirts','Sweaters','Shirts','Tank Tops','Water Bottles','Other Items (#)']

# # Multiply each category count by the condition counts per row, then sum across all rows
# use_contingency = pd.DataFrame({
#     'New/Like-New': df[categories].mul(df['Condition (New/Like-New)'], axis=0).sum(),
#     'Moderately-Used': df[categories].mul(df['Condition (Moderately-Used)'], axis=0).sum()
# })
# #Removing Empty Rows:
# use_contingency = use_contingency[(use_contingency.T != 0).any()]

# print(use_contingency)

In [16]:
# chi2, p, dof, expected = chi2_contingency(use_contingency)
# print(f"Chi2 statistic: {chi2}")
# print(f"p-value: {p}")